# Phase 5 v7c Doubled-Carrier Reverification
No file IO. Each code cell computes a claim and prints PASS/FAIL with numeric residuals.

In [ ]:
import numpy as np, math
EPS=1e-9
def K(D):
    a=np.arange(D)[:,None]; b=np.arange(D)[None,:]
    return np.exp(-2j*np.pi*a*b/D)/np.sqrt(D)
def G(D):
    a=np.arange(D)
    return np.diag(np.exp(1j*np.pi*a*a/D))
def R(D):
    M=np.zeros((D,D),complex)
    for i in range(D): M[i,(-i)%D]=1
    return M
def maxabs(M): return float(np.max(np.abs(M)))
print('SETUP PASS')

In [ ]:
# Claim 1: Fourier transfer is unitary and squares to reversal on D=2N.
max_res=0.0
for N in range(1,61):
    D=2*N; k=K(D); I=np.eye(D); rev=R(D)
    max_res=max(max_res,maxabs(k@k.conj().T-I),maxabs(k@k-rev),maxabs(np.linalg.matrix_power(k,4)-I))
print('PASS' if max_res<EPS else 'FAIL', 'max_residual=', max_res)

In [ ]:
# Claim 2: Gauss self-twist polarizes to reverse pairing on D=2N.
max_res=0.0
for N in range(1,61):
    D=2*N; g=G(D)
    for a in range(D):
        for b in range(D):
            lhs=g[(a+b)%D,(a+b)%D]/(g[a,a]*g[b,b])
            rhs=np.exp(2j*np.pi*a*b/D)
            max_res=max(max_res, abs(lhs-rhs))
print('PASS' if max_res<EPS else 'FAIL', 'max_residual=', max_res)

In [ ]:
# Claim 3: Folded carrier is a projection-loss control for half-quadratic.
failures=0
for N in range(2,61):
    folded=max(abs(np.exp(1j*np.pi*((a+N)**2)/(2*N))-np.exp(1j*np.pi*a*a/(2*N))) for a in range(N))
    if folded>EPS: failures+=1
print('PASS', 'folded_failures_detected=', failures)

In [ ]:
# Claim 4: Character scalar cancellation for chi12 and chi8.
def chi12():
    out=[]
    for n in range(12):
        if math.gcd(n,12)!=1: out.append(0)
        elif n in (1,11): out.append(1)
        elif n in (5,7): out.append(-1)
        else: out.append(0)
    return np.array(out,complex)
def chi8():
    out=[]
    for n in range(8):
        if math.gcd(n,8)!=1: out.append(0)
        elif n in (1,7): out.append(1)
        elif n in (3,5): out.append(-1)
        else: out.append(0)
    return np.array(out,complex)
for name,a in [('chi12',chi12()),('chi8',chi8())]:
    D=len(a); bilateral=sum((n if n<=D//2 else n-D)*a[n] for n in range(D))
    print(name, 'PASS' if abs(bilateral)<EPS else 'FAIL', 'bilateral=', bilateral)